# Train a timing side-channel model in Colab

Collect synthetic measurements with the Windows native implementation first. This notebook trains on portable derived features; it cannot collect Windows counter measurements. A CPU runtime is sufficient and no packages need installing.

On Windows, run:
```powershell
python research_pipeline.py export --dataset datasets/natural-v1 --output datasets/natural-v1-colab.zip
```
Upload that ZIP below. Synthetic secrets in its manifest are orchestration metadata, never model inputs. The split was fixed before collection. Training reads only train/validation feature files; test evaluation remains on Windows.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one exported training ZIP.'
archive_path = next(iter(uploaded))


In [ ]:
from pathlib import Path
import zipfile
import os
workspace = Path('/content/timing_research')
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(workspace)
os.chdir(workspace)
import json
manifest = json.loads(Path('dataset/manifest.json').read_text())
print('Unique passwords by split:', manifest['split_counts'])
print('Collection config:', manifest['config'])
print('Feature groups per session are split by whole password, never random CSV rows.')


## Fit and choose regularization on validation data

This fits a linear group-softmax model with L2 candidates 0, 0.001 and 0.01. It selects validation password-macro character accuracy, with cross-entropy as a tiebreaker. Keep the test set untouched while choosing the model. Character recovery during calibration uses known correct prefixes; blind recovery is a separate, harder evaluation.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, 'research_pipeline.py', 'train',
                '--dataset', 'dataset', '--output', 'model.json',
                '--epochs', '50', '--seed', '7'], check=True)
model = json.loads(Path('model.json').read_text())
print('Selected L2:', model['l2'])
print('Test used for selection:', model['test_used_for_selection'])
for experiment in model['validation_experiments']:
    overall = experiment['validation']['buckets']['all']
    print(experiment['l2'], 'model accuracy:', overall['model_accuracy'],
          'median accuracy:', overall['median_accuracy'])


## Download the model; evaluate fresh measurements on Windows

Place the downloaded model in `models/colab-natural-v1.json` in your original project, then run:
```powershell
python research_pipeline.py evaluate --dataset datasets/natural-v1 --model models/colab-natural-v1.json --output research_results/colab-natural-v1
```
The manifest fingerprint must match. Evaluation reports both held-out calibration accuracy and fresh blind recovery for model/median, vulnerable/control. Do not change the model based on test results and reuse that same test set as if it were untouched.


In [ ]:
files.download('model.json')
